# Experimentación — Calibración de frases para embeddings
### Metodología: revisar fragmentos reales para afinar `GRUPOS_EMBEDDINGS`

Este notebook es un **taller de calibración**, no un análisis final. Sirve para leer fragmentos reales de las mañaneras y decidir si las frases descriptivas de cada grupo (en `Analisis_Embeddings.ipynb`) están funcionando bien.

Flujo de trabajo sugerido (normalmente 2–3 vueltas):

1. Corre las celdas de CONFIGURACIÓN y CÓDIGO BASE.
2. Corre el bloque del grupo que te interese.
3. Lee los **top fragmentos por similitud** — ¿de verdad hablan del grupo? Si no, tus frases están jalando ruido.
4. Lee los **fragmentos cerca del umbral** — ¿deberían haber contado o no? Si el corte se siente mal, ajusta el umbral o las frases.
5. Edita `GRUPOS_EMBEDDINGS` en la celda de configuración, vuelve a correr desde "Cálculo de embeddings" en adelante, y repite el bloque del grupo que ajustaste.
6. Cuando quede bien, copia las frases finales a `Analisis_Embeddings.ipynb`.

> Requiere las mismas dependencias que `Analisis_Embeddings.ipynb` (`requirements-embeddings.txt`).

---
## CONFIGURACIÓN

In [2]:
# =============================================================
# CONFIGURACIÓN
# =============================================================

# Carpeta local donde están guardados tus archivos .docx
RUTA_DATOS = "./data/mananeras"

# Rango de fechas del análisis
FECHA_INICIO = "2018-12-01"   # formato: AAAA-MM-DD
FECHA_FIN    = "2026-01-31"   # formato: AAAA-MM-DD

# Corte entre presidencias
FECHA_CAMBIO_PRESIDENTE = "2024-10-01"

# =============================================================
# GRUPOS Y FRASES DESCRIPTIVAS PARA EMBEDDINGS SEMÁNTICOS
# Copia/pega aquí las mismas frases que tengas en Analisis_Embeddings.ipynb
# para experimentar, y cuando queden bien cópialas de vuelta.
# =============================================================

GRUPOS_EMBEDDINGS = {
    "Periodistas": [
        "periodistas y prensa libre en México",
        "medios de comunicación y reporteros",
        "libertad de prensa y protección a periodistas",
        "comunicadores, corresponsales y periodistas de investigación",
        "agresiones a periodistas y libertad de expresión"
    ],

    "Ambientalistas": [
        "ambientalistas y ecologistas mexicanos",
        "protección del medio ambiente y sustentabilidad",
        "cambio climático y activismo ambiental",
        "defensa de los ecosistemas y la biodiversidad",
        "defensores del territorio y recursos naturales"
        "contaminación y mal manejo de residuos"
    ],

    "Colectivos_victimas": [
        "colectivos de víctimas y familiares de desaparecidos",
        "madres buscadoras y personas desaparecidas",
        "búsqueda de personas desaparecidas y fosas clandestinas",
        "verdad, justicia y reparación para las víctimas",
        "grupos afectados por la violencia"
    ],

    "ONGs": [
        "organizaciones de la sociedad civil y ONG",
        "asociaciones civiles sin fines de lucro",
        "organizaciones defensoras de derechos humanos",
        "organizaciones no gubernamentales y participación ciudadana",
        "sociedad civil organizada y organizaciones civiles"
    ],
}

# Umbral de similitud coseno para considerar un fragmento "relevante"
UMBRAL_SIMILITUD = 0.6

# Qué tan cerca del umbral se considera "zona de duda" a revisar
VENTANA_UMBRAL = 0.05

# Cuántos fragmentos mostrar en cada bloque de inspección
TOP_N = 10

print("✅ Config guardada.")
print(f"   Período: {FECHA_INICIO} → {FECHA_FIN}")
print(f"   Grupos: {list(GRUPOS_EMBEDDINGS.keys())}")
print(f"   Umbral: {UMBRAL_SIMILITUD}  (ventana de duda: ±{VENTANA_UMBRAL})")


✅ Config guardada.
   Período: 2018-12-01 → 2026-01-31
   Grupos: ['Periodistas', 'Ambientalistas', 'Colectivos_victimas', 'ONGs']
   Umbral: 0.6  (ventana de duda: ±0.05)


---
## CÓDIGO BASE — No modificar

### Carga de archivos locales (.docx)

In [3]:
import os, re
from datetime import datetime
from docx import Document
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Parsear fechas de configuración
_fecha_inicio = datetime.strptime(FECHA_INICIO, "%Y-%m-%d")
_fecha_fin    = datetime.strptime(FECHA_FIN,    "%Y-%m-%d")
_fecha_cambio = datetime.strptime(FECHA_CAMBIO_PRESIDENTE, "%Y-%m-%d")

_MESES_ES = {
    "ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12
}

def _extraer_fecha(nombre):
    m = re.search(r'([A-Za-z]{3})[-_](\d{4})', nombre)
    if m:
        mes_str = m.group(1).upper()
        año = int(m.group(2))
        mes = _MESES_ES.get(mes_str)
        if mes:
            return datetime(año, mes, 1)
    return None

def _cargar_corpus():
    if not os.path.exists(RUTA_DATOS):
        print(f"⚠️  Carpeta no encontrada: {RUTA_DATOS}")
        print("   Verifica que RUTA_DATOS apunte a la carpeta correcta en tu computadora.")
        return []
    archivos = []
    sin_fecha = []
    for nombre in sorted(os.listdir(RUTA_DATOS)):
        if not nombre.lower().endswith('.docx'):
            continue
        fecha = _extraer_fecha(nombre)
        if not fecha:
            sin_fecha.append(nombre)
            continue
        if not (_fecha_inicio <= fecha <= _fecha_fin):
            continue
        ruta = os.path.join(RUTA_DATOS, nombre)
        try:
            doc = Document(ruta)
            texto = ' '.join(p.text.strip() for p in doc.paragraphs if p.text.strip())
            presidente = "AMLO" if (fecha.year, fecha.month) < (_fecha_cambio.year, _fecha_cambio.month) else "Claudia Sheinbaum"
            archivos.append({
                'fecha':      fecha,
                'mes':        fecha.strftime('%Y-%m'),
                'año':        fecha.year,
                'presidente': presidente,
                'texto':      texto,
                'n_palabras': len(texto.split())
            })
        except Exception as e:
            print(f"   ⚠️  Error leyendo {nombre}: {e}")

    print(f"✅ {len(archivos)} archivos cargados")
    print(f"   Período: {_fecha_inicio.strftime('%b %Y')} → {_fecha_fin.strftime('%b %Y')}")
    dist = pd.Series([d['presidente'] for d in archivos]).value_counts()
    for p, n in dist.items():
        print(f"   {p}: {n} archivos")
    if sin_fecha:
        print(f"\n   ⚠️  {len(sin_fecha)} archivos ignorados (no se pudo leer fecha del nombre):")
        for f in sin_fecha[:5]:
            print(f"      {f}")
    return sorted(archivos, key=lambda x: x['fecha'])

_corpus = _cargar_corpus()


✅ 86 archivos cargados
   Período: Dec 2018 → Jan 2026
   AMLO: 70 archivos
   Claudia Sheinbaum: 16 archivos


### Cálculo de embeddings y similitudes a nivel de fragmento

A diferencia de `Analisis_Embeddings.ipynb` (que agrega resultados por documento), aquí guardamos **cada fragmento individual con su texto y su similitud por grupo**, para poder leerlos uno por uno.

In [ ]:
# ============================================================
#  CÁLCULO DE EMBEDDINGS A NIVEL DE FRAGMENTO
# ============================================================

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

if not _corpus:
    print("No hay archivos cargados. Revisa RUTA_DATOS y vuelve a ejecutar la celda de carga.")
else:
    print("Cargando modelo de embeddings (paraphrase-multilingual-MiniLM-L12-v2)...")
    _modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("✅ Modelo listo.\n")

    print("Generando vectores de grupos de interés...")
    _emb_grupos = {}
    for grupo, frases in GRUPOS_EMBEDDINGS.items():
        vecs = _modelo.encode(frases, show_progress_bar=False)
        _emb_grupos[grupo] = np.mean(vecs, axis=0)
        print(f"   ✓ {grupo}")

    def _chunking(texto, n=3):
        # Dividir el texto en oraciones y luego agruparlas en fragmentos de n oraciones
        #También se filtran oraciones muy cortas (<40 caracteres) y fragmentos muy cortos (<60 caracteres)
        oraciones = [s.strip() for s in re.split(r'(?<=[.!?])\s+', texto) if len(s.strip()) > 40]
        return [' '.join(oraciones[i:i+n]) for i in range(0, len(oraciones), n)
                if len(' '.join(oraciones[i:i+n])) > 60]

    print(f"\nProcesando {len(_corpus)} archivos (guardando cada fragmento)...\n")
    filas_chunks = []

    for i, doc in enumerate(_corpus):
        if (i+1) % 30 == 0 or i == 0:
            print(f"   Archivo {i+1}/{len(_corpus)} — {doc['fecha'].strftime('%b %Y')} ({doc['presidente']})")
        chunks = _chunking(doc['texto'])
        if not chunks:
            continue
        emb_chunks = _modelo.encode(chunks, batch_size=64, show_progress_bar=False)

        for grupo, emb_g in _emb_grupos.items():
            sims = cosine_similarity([emb_g], emb_chunks)[0]
            for texto_frag, sim in zip(chunks, sims):
                filas_chunks.append({
                    'fecha':      doc['fecha'],
                    'mes':        doc['mes'],
                    'presidente': doc['presidente'],
                    'grupo':      grupo,
                    'similitud':  round(float(sim), 4),
                    'fragmento':  texto_frag
                })

    df_chunks = pd.DataFrame(filas_chunks)
    print(f"\n✅ {len(df_chunks)} fragmentos procesados en total (documentos × chunks × grupos).")


Cargando modelo de embeddings (paraphrase-multilingual-MiniLM-L12-v2)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10175.88it/s]


✅ Modelo listo.

Generando vectores de grupos de interés...
   ✓ Periodistas
   ✓ Ambientalistas
   ✓ Colectivos_victimas
   ✓ ONGs

Procesando 86 archivos (guardando cada fragmento)...

   Archivo 1/86 — Dec 2018 (AMLO)
   Archivo 30/86 — May 2021 (AMLO)
   Archivo 60/86 — Nov 2023 (AMLO)

✅ 1021972 fragmentos procesados en total (documentos × chunks × grupos).


### Función de inspección (reutilizable para cualquier grupo)

In [6]:
# ============================================================
#  FUNCIÓN DE INSPECCIÓN POR GRUPO
# ============================================================

def inspeccionar_grupo(grupo, top_n=TOP_N, ventana=VENTANA_UMBRAL):
    """
    Muestra, para un grupo de GRUPOS_EMBEDDINGS:
      1) Los `top_n` fragmentos con mayor similitud (para detectar ruido / falsos positivos)
      2) Los fragmentos dentro de ±`ventana` del umbral (para calibrar el corte)

    Devuelve el DataFrame completo de ese grupo, ordenado por similitud descendente,
    por si quieres explorarlo tú mismo (df.head(30), df.to_csv(...), etc).
    """
    dg = df_chunks[df_chunks['grupo'] == grupo].sort_values('similitud', ascending=False)

    print("═"*70)
    print(f"  {grupo.upper()} — TOP {top_n} FRAGMENTOS POR SIMILITUD")
    print("  (¿de verdad hablan del grupo, o tus frases están jalando ruido?)")
    print("═"*70)
    for _, row in dg.head(top_n).iterrows():
        texto_corto = row['fragmento'][:300] + ("..." if len(row['fragmento']) > 300 else "")
        print(f"\n[{row['fecha'].strftime('%Y-%m-%d')} | {row['presidente']} | sim={row['similitud']:.3f}]")
        print(f"  {texto_corto}")

    cerca = dg[(dg['similitud'] >= UMBRAL_SIMILITUD - ventana) &
               (dg['similitud'] <= UMBRAL_SIMILITUD + ventana)].sort_values('similitud', ascending=False)

    print("\n" + "═"*70)
    print(f"  {grupo.upper()} — FRAGMENTOS CERCA DEL UMBRAL ({UMBRAL_SIMILITUD} ± {ventana})")
    print("  (¿deberían haber contado o no? Aquí se calibra el corte)")
    print("═"*70)
    print(f"  ({len(cerca)} fragmentos en esta ventana)")
    for _, row in cerca.head(top_n).iterrows():
        marca = "✅ cuenta" if row['similitud'] >= UMBRAL_SIMILITUD else "❌ no cuenta"
        texto_corto = row['fragmento'][:300] + ("..." if len(row['fragmento']) > 300 else "")
        print(f"\n[{row['fecha'].strftime('%Y-%m-%d')} | {row['presidente']} | sim={row['similitud']:.3f} | {marca}]")
        print(f"  {texto_corto}")

    print(f"\n✅ Inspección de '{grupo}' completada.\n")
    return dg


---
## Grupos

Correr la función inspeccionar para cada uno de los grupos

In [7]:
_df_periodistas = inspeccionar_grupo("Periodistas")


══════════════════════════════════════════════════════════════════════
  PERIODISTAS — TOP 10 FRAGMENTOS POR SIMILITUD
  (¿de verdad hablan del grupo, o tus frases están jalando ruido?)
══════════════════════════════════════════════════════════════════════

[2025-11-01 | Claudia Sheinbaum | sim=0.808]
  Los medios tienen la responsabilidad de poner las dos posiciones siempre, de debatir, de discutir, no solo una, ¿verdad?, porque no que sea unos cuantos. O utilizan sus periódicos o sus medios como propaganda política. Bueno, entonces pónganle: “Órgano oficial de la derecha mexicana” PREGUNTA: Justa...

[2022-01-01 | AMLO | sim=0.807]
  INTERLOCUTORA: Sobre todo, bueno, porque en pocos días también se dio el asesinato de otro periodista, Margarito. Y, bueno, la indignación en el gremio y el cuestionamiento de qué tiene que pasar. Si bien es cierto, no es solamente en los últimos años ni privativo de este sexenio, pero ¿qué tiene qu...

[2021-08-01 | AMLO | sim=0.807]
  Desde este espaci

In [8]:
_df_ambientalistas = inspeccionar_grupo("Ambientalistas")


══════════════════════════════════════════════════════════════════════
  AMBIENTALISTAS — TOP 10 FRAGMENTOS POR SIMILITUD
  (¿de verdad hablan del grupo, o tus frases están jalando ruido?)
══════════════════════════════════════════════════════════════════════

[2024-01-01 | AMLO | sim=0.771]
  (INICIA VIDEO) VOZ MUJER: En el gobierno de la Cuarta Transformación trabajamos para el bienestar de las personas y hacemos historia en el cuidado del medioambiente con el decreto de 43 nuevas áreas naturales protegidas. Sumamos más de tres millones de hectáreas que conservan mares, playas, bosques,...

[2023-04-01 | AMLO | sim=0.768]
  MARÍA LUISA ALBORES GONZÁLEZ, SECRETARIA DE MEDIO AMBIENTE Y RECURSOS NATURALES: Buenos días a todas y a todos. Voy a hablar de los avances ambientales, ahora me voy a dedicar a platicar un poco de lo que estamos haciendo en este tramo con áreas naturales protegidas. Estas son áreas naturales proteg...

[2023-12-01 | AMLO | sim=0.765]
  Pero que, además, ese terre

In [9]:
_df_ambientalistas = inspeccionar_grupo("Colectivos_victimas")


══════════════════════════════════════════════════════════════════════
  COLECTIVOS_VICTIMAS — TOP 10 FRAGMENTOS POR SIMILITUD
  (¿de verdad hablan del grupo, o tus frases están jalando ruido?)
══════════════════════════════════════════════════════════════════════

[2025-03-01 | Claudia Sheinbaum | sim=0.725]
  Pero tenemos que decir la verdad, cuántas personas desaparecidas existen en la plataforma de la Comisión Nacional de Búsqueda, y cuál es el origen de la información. Los familiares de las personas que buscan a sus hijos, pues siempre va a haber cercanía y atención a las víctimas. Entonces, eso hay q...

[2022-06-01 | AMLO | sim=0.720]
  También informar que todas las diligencias, entrevistas, actuaciones, peritajes que se han venido realizando, se pudo constatar por este grupo que ya obran dentro de la carpeta de investigación, a la cual han tenido acceso desde luego los padres de la víctima y también la Comisión de Atención a Víct...

[2021-01-01 | AMLO | sim=0.717]
  En materi

In [10]:
_df_ambientalistas = inspeccionar_grupo("ONGs")


══════════════════════════════════════════════════════════════════════
  ONGS — TOP 10 FRAGMENTOS POR SIMILITUD
  (¿de verdad hablan del grupo, o tus frases están jalando ruido?)
══════════════════════════════════════════════════════════════════════

[2021-08-01 | AMLO | sim=0.743]
  Es la primera vez que vamos a hacer un convenio con una organización civil, una organización de la sociedad civil, porque habíamos decidido entregar todo de manera directa a los beneficiarios sin intermediación y se acostumbraba a entregar apoyos del gobierno, presupuesto a asociaciones civiles y mu...

[2022-01-01 | AMLO | sim=0.718]
  Ayer hablaba yo, ya no está ese instituto de desarrollo social o de apoyo a la justicia, pero está la Secretaría de Bienestar. Pues era para entregarle dinero a las llamadas organizaciones no gubernamentales y por lo general las organizaciones no gubernamentales, de la sociedad civil —y ese es un bu...

[2020-01-01 | AMLO | sim=0.707]
  PRESIDENTE ANDRÉS MANUEL LÓPEZ OBRADO